# Customer Outreach Campaign

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Using python3.11.3

!pip3 install -r ../requirements.txt

/Users/namitjain/.zshenv:.:1: no such file or directory: /Users/namitjain/.cargo/env


In [4]:
from crewai import Agent, Task, Crew

In [5]:
import os
from utils import get_openai_api_key, pretty_print_result
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [6]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at Arcadia, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [10]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of Arcadia's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

In [11]:
from crewai_tools import DirectoryReadTool, FileReadTool, SerperDevTool

In [12]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [14]:
from crewai.tools import BaseTool

In [15]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [16]:
sentiment_analysis_tool = SentimentAnalysisTool()

In [17]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

In [18]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

In [20]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

In [21]:
inputs = {
    "lead_name": "CrewAI",
    "industry": "AI Agent Framework",
    "key_decision_maker": "João (Joe) Moura",
    "position": "Founder",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

# Agent: Sales Representative
## Task: Conduct an in-depth analysis of CrewAI, a company in the AI Agent Framework sector that recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile, focusing on key decision-makers, recent business developments, and potential needs that align with our offerings. This task is crucial for tailoring our engagement strategy effectively.
Don't make assumptions and only use information you absolutely sure about.


# Agent: Sales Representative
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"CrewAI company background key personnel recent developments 2023\"}"
## Tool Output: 
{'searchParameters': {'q': 'CrewAI company background key personnel recent developments 2023', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'CrewAI 2025 Company Profile: Valuation, Funding & Investors', 'link': 'https://pitchbook.com/profiles/company/590845-78', 'snippet':

In [25]:
from IPython.display import Markdown
# Markdown(result.raw)

Markdown(pretty_print_result(result.raw))

Email Campaign Drafts for João Moura, Founder of CrewAI:

---

**Email 1: Introducing Our Synergistic Solutions**

Subject: Empowering CrewAI's Vision with Our Scalable Solutions

Hi João,

Congratulations on CrewAI's phenomenal success in securing $18 million in Series
A funding and the successful launch of your enterprise cloud platform! It's
exciting to see such a demand with over 40,000 companies eager to integrate your
innovative AI solutions.

At [Your Company Name], we admire your vision of empowering businesses through
state-of-the-art multi-agent systems. We understand that scaling up rapidly to
meet client demand while maintaining seamless integration is critical at this
stage of your growth.

To support CrewAI's ambitious journey, we'd love to offer a complimentary
demonstration of our scalable AI solutions. Our technology not only enhances
functionalities but is designed to integrate effortlessly with existing
platforms, ensuring smooth adaptations to increasing client needs.

Furthermore, in line with your move towards cloud solutions, cybersecurity is
paramount. Our security services can play a vital role in safeguarding your data
and maintaining compliance, which is crucial for your enterprise clients.

We also recognize the importance of providing exceptional customer experiences,
especially with a large waitlist. Our CRM tools can streamline onboarding and
support processes, helping you achieve high client satisfaction from the outset.

Would you be open to a brief call or meeting to explore how our tailored
solutions can seamlessly align with CrewAI's strategy and vision?

Looking forward to your thoughts.

Best regards,

[Your Full Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]

---

**Email 2: Enhancing Cloud Security and Integration**

Subject: Let's Fortify CrewAI's Cloud Security

Hi João,

I hope this message finds you well. Following your recent strides in the AI
sector, I believe our security solutions could provide substantial value to
CrewAI's enterprise cloud platform. Our latest offerings focus on advanced data
protection and compliance—key essentials as you scale your client base.

Our team is keen on offering a detailed cybersecurity audit, helping identify
and mitigate potential vulnerabilities in your existing systems. With our
support, you can ensure seamless and secure operations for your rapidly growing
clientele.

This partnership could position CrewAI to not only meet but exceed the top-tier
security expectations of your enterprise clients.

Are you available for a quick discussion on how we can bolster CrewAI's security
framework?

Thank you for considering this opportunity.

Warm regards,

[Your Full Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]

---

**Email 3: Enhancing Client Onboarding Experience**

Subject: Elevate Your Client Onboarding with Our CRM Solutions

Dear João,

With CrewAI's impressive waitlist, delivering an unparalleled onboarding
experience will be crucial for maintaining your leadership stance in the AI
field. At [Your Company Name], we specialize in CRM solutions that simplify and
enhance client interactions, ensuring every new user feels valued and informed
from their first interaction.

Our tools are designed to streamline the onboarding process, enabling your team
to focus on delivering innovation rather than administrative tasks. We would be
thrilled to provide a demonstration tailored specifically for CrewAI.

Could we arrange a time next week to show you how our CRM systems can elevate
CrewAI's customer experience?

Excited about the possibility of working together.

Best wishes,

[Your Full Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]

---

These drafts aim to establish a collaborative narrative that aligns our
solutions with CrewAI's recent accomplishments and future objectives, fostering
an engaging and authentic connection.
```